# L6-3c Demo: Action space matters

This notebook recreates the Lecture 3 action-interface comparison in a crawler-scale toy problem. The question is not whether we can replicate the full humanoid experiment from `pengvanderpanne2017action`, but whether the *qualitative ordering* survives:

`pd_target_with_gains` >= `pd_target` >> `torque`


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from l6_3_utils import (
    CMM_AMBER,
    CMM_BLUE,
    CMM_ROSE,
    CMM_SLATE,
    FIGURE_DIR,
    save_figure,
    smooth_xy,
    train_or_load,
)

TOTAL_TIMESTEPS = 50_000
SEED = 13

action_envs = {
    "torque": dict(include_velocity=True, action_mode="torque", reward_mode="dense_vel"),
    "pd_target": dict(include_velocity=True, action_mode="pd_target", reward_mode="dense_vel"),
    "pd_target_with_gains": dict(
        include_velocity=True,
        action_mode="pd_target_with_gains",
        reward_mode="dense_vel",
    ),
}

print(f"Saving lecture figures into: {FIGURE_DIR}")


In [ ]:
torque_artifact = train_or_load(
    "ppo",
    "l6_3c_ppo_torque.zip",
    action_envs["torque"],
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)
pd_target_artifact = train_or_load(
    "ppo",
    "l6_3c_ppo_pd_target.zip",
    action_envs["pd_target"],
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)
pd_gains_artifact = train_or_load(
    "ppo",
    "l6_3c_ppo_pd_target_with_gains.zip",
    action_envs["pd_target_with_gains"],
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)

print("Recent torque reward:", np.mean(torque_artifact.rewards[-5:]))
print("Recent PD target reward:", np.mean(pd_target_artifact.rewards[-5:]))
print("Recent PD+gains reward:", np.mean(pd_gains_artifact.rewards[-5:]))


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 2.8))

for label, artifact, color in [
    ("torque", torque_artifact, CMM_ROSE),
    ("PD target", pd_target_artifact, CMM_AMBER),
    ("PD target + gains", pd_gains_artifact, CMM_BLUE),
]:
    x, y = smooth_xy(artifact.steps, artifact.rewards, window=5)
    ax.plot(x, y, lw=2.0, color=color, label=label)

ax.set_xlabel("environment steps")
ax.set_ylabel("episode reward")
ax.set_title("Crawler PPO with three action interfaces", color=CMM_SLATE)
ax.grid(True, alpha=0.25)
ax.legend(loc="lower right", fontsize=9)

fig_path = save_figure(fig, "action_space_curves.pdf")
fig_path


Assignment 3 makes the same design choice for a reason: `Assignments/cmm-26-a3/animRL/cfg/mimic/mimic_pi_config.py` uses `control_type='P'`, which is the fixed-gain PD-target middle option in this toy comparison rather than raw torque control.
